# Deep Learning Architecture Simulator
**From ordinary neurons to CNNs, RNNs, and multimodal fusion.** This is a companion to Neural Network Simulator. Run top-to-bottom after `pip install -e ".[notebook]"` in the repository root.

**Important:** a CNN channel is a learned *feature map*; it is not a general-purpose executable Python algorithmic unit. CNNs reuse spatial filters, and RNNs reuse a recurrent cell across sequence steps.

## 1. A conventional neuron and a learnable layer
A single neuron computes $y=\sigma(w^Tx+b)$. `nn.Linear` computes many neurons in parallel. The layer's parameters are updated by backpropagation.

In [ ]:
import torch
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dl_arch_sim import ModelConfig, DeepLearningSimulator, make_demo_data, train

torch.manual_seed(13)
x = torch.tensor([[0.2, 0.5, -0.3]])
layer = nn.Sequential(nn.Linear(3, 4), nn.ReLU())
print("Dense layer", tuple(x.shape), "→", tuple(layer(x).shape))

## 2. Select the architecture and compute dimensions
- **CNN:** image tensors `[batch, channels, height, width]`.
- **RNN:** time-series `[batch, timesteps, features]`.
- **Hybrid:** one network with CNN and RNN branches, concatenate their latent representations, then train a common prediction head.

Adjust the config cell and rerun.

In [ ]:
cfg = ModelConfig(
    architecture="hybrid",  # "cnn", "rnn", "hybrid"
    cnn_channels=8,
    rnn_hidden=16,
    rnn_layers=1,
    cell="gru",  # "gru", "lstm", "rnn"
    seq_len=12,
    image_size=16,
    rnn_features=4
)
model = DeepLearningSimulator(cfg)
inputs, target = make_demo_data(samples=240, image_size=cfg.image_size,
    image_channels=cfg.image_channels, seq_len=cfg.seq_len, seq_features=cfg.rnn_features)
print(model)
print(f"Trainable parameters: {model.parameter_count():,}")
print("Image:", tuple(inputs['image'].shape))
print("Sequence:", tuple(inputs['sequence'].shape))
print("Fused features:", tuple(model.features(inputs[:1]) .shape) if False else tuple(model.features({k:v[:1] for k,v in inputs.items()}).shape))
print("Predictions:", tuple(model({k:v[:8] for k,v in inputs.items()}).shape))

## 3. Visualize the inputs
The toy target depends on spatial information from the image center **and** recent sequence values, so the hybrid has access to more relevant evidence than either single-modality model.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
sns.heatmap(inputs['image'][0,0].numpy(), cmap="viridis", ax=ax[0], cbar=False)
ax[0].set_title("One spatial image")
for f in range(cfg.rnn_features):
    ax[1].plot(inputs['sequence'][0,:,f].numpy(), label=f"feature {f}")
ax[1].set_title("Sequence over time")
ax[1].legend(ncol=2, fontsize=8)
plt.tight_layout()

## 4. Train all three architectures side by side
Same labeled synthetic dataset and train/validation split seed. This illustrates architecture behavior rather than establishing which model is best in general.

In [ ]:
from dataclasses import replace
results = {}
models = {}
for architecture in ["cnn", "rnn", "hybrid"]:
    torch.manual_seed(13)
    m = DeepLearningSimulator(replace(cfg, architecture=architecture))
    history = train(m, inputs, target, epochs=18, learning_rate=0.01)
    results[architecture] = history
    models[architecture] = m
    print(architecture, "parameters:", m.parameter_count(),
          "validation accuracy:", round(history['validation_accuracy'][-1], 3))

In [ ]:
rows = []
for name, history in results.items():
    for epoch, (loss, acc) in enumerate(zip(history['validation_loss'], history['validation_accuracy']), start=1):
        rows.append({"architecture":name,"epoch":epoch,"validation loss":loss,"validation accuracy":acc})
df = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df, x="epoch", y="validation loss", hue="architecture", ax=axes[0])
sns.lineplot(data=df, x="epoch", y="validation accuracy", hue="architecture", ax=axes[1])
plt.tight_layout()

## 5. Under the hood: feature maps and hidden states
A convolution returns feature maps `[B,C,H,W]`; with more filters, more output channels. The recurrent branch returns a hidden state `[B,H]`. In the hybrid, the vectors are concatenated and sent to a learned dense head.

In [ ]:
with torch.no_grad():
    sample = {k:v[:2] for k,v in inputs.items()}
    if models['hybrid'].cnn is not None:
        conv1 = models['hybrid'].cnn.layers[0](sample['image'])
        print("First CNN convolution:", tuple(conv1.shape))
    _, recurrent_state = models['hybrid'].rnn.recurrent(sample['sequence'])
    if isinstance(recurrent_state, tuple): recurrent_state = recurrent_state[0]
    print("All RNN layer hidden states:", tuple(recurrent_state.shape))
    print("Fused representation:", tuple(models['hybrid'].features(sample).shape))

## 6. Industry and problem starting points
These are initial design hypotheses. Compare with simple baselines before putting into production.

In [ ]:
problems = {
    "finance / sequential anomalies": "rnn",
    "healthcare / image research": "cnn",
    "entertainment / visuals + event history": "hybrid",
    "manufacturing / image + sensor sequence": "hybrid",
    "science / spatiotemporal observations": "hybrid",
}
print(pd.DataFrame([{"problem":k,"initial architecture":v} for k,v in problems.items()]).to_string(index=False))

## 7. Backends and scaling
Try CPU or CUDA when available. Moving both models and input tensors to the selected device matters. Measure throughput before adding custom C++/CUDA kernels. Distinguish a *trainable CNN channel* from a *plug-in C++ computational unit* in the other simulator.

**Next extensions:** datasets and encoders, model registry, FastAPI serving, Temporal CICDCT, Helm, and compute estimator. None are claimed as implemented here.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available device:", device)
model_on_device = models["hybrid"].to(device).eval()
with torch.no_grad():
    batch = {k:v[:8].to(device) for k,v in inputs.items()}
    probabilities = model_on_device(batch).sigmoid()
print("Predictions:", probabilities.cpu().round(decimals=3).tolist())